# ⚡ KOSIS RAG 50문항 2단계(Batch JSON 저장 ➔ 초고속 Human 검수 카드) 워크벤치

이 노트북은 **자동 실행 단계(1단계)**와 **사람의 시각적 검수 단계(2단계)**를 완전히 분리하여, 사람이 50문항의 **질문·AI 답변·실측 근거 표·KOSIS 링크·5대 자동 지표 배지**만 한눈에 보고 **1초 만에 `PASS` / `PARTIAL` / `FAIL`과 코멘트를 기록**할 수 있도록 설계되었습니다.

--- 
### 📌 2단계 워크플로우
- **[Phase 1] 1차 자동 배치 조회 및 근거 자료 포함 JSON 저장 (`[Cell 2]`)**
  - `kosis_rag_eval_50.csv`의 50개 쿼리를 모두 조회하여 **AI 답변, 5대 자동화 지표(`PASS`/`PARTIAL`/`FAIL`), 그리고 판단 근거가 되는 원본 데이터(`facts`, 조건 적용 표 `condition_rows`, 미리보기 표 `preview`, KOSIS URL, 로컬 CSV 경로)**를 통째로 `eval_50_auto_results.json`에 저장합니다.
- **[Phase 2] 사람이 보기 편한 시각적 검수 카드 출력 & 초간편 판정 입력 (`[Cell 3]` ~ `[Cell 5]`)**
  - 저장된 `eval_50_auto_results.json`을 불러와 **컬러 배지(`🟢 PASS`, `🟡 PARTIAL`, `🔴 FAIL`)가 적용된 대시보드와 근거 비교 카드**를 화면에 깔끔하게 렌더링합니다.
  - 사람은 출력된 카드만 보고 `[Cell 5]`(인터랙티브 위젯 또는 간편 딕셔너리 입력)에서 바로 최종 평가(`PASS` / `PARTIAL` / `FAIL`)와 한 줄 텍스트만 적어 즉시 저장합니다.

In [1]:
# [Cell 1] 환경 설정 및 모듈 로드
import json
import os
import re
import time
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, HTML, Markdown, clear_output
import pandas as pd

# .env 리로드 및 API 키 공백/따옴표 제거 (HTTP 401 에러 방지)
load_dotenv(Path.cwd() / ".env", override=True)
if os.getenv("HCX_API_KEY"):
    os.environ["HCX_API_KEY"] = os.getenv("HCX_API_KEY").strip().strip("'\"")

import kosis_views as kv
import kosis_rag_qa as qa

DATA_DIR = Path("kosis_data")
# VIEWS_DIR = Path("kosis_data/rag_views_golden") if Path("kosis_data/rag_views_golden").exists() else Path("kosis_data/rag_views")
# VARIANT = "golden_v1" if (DATA_DIR / "index" / "golden_v1" / "meta.json").exists() else "base"
VIEWS_DIR = Path("kosis_data/rag_views")
VARIANT = "base"
qa.VIEWS_DIR = VIEWS_DIR
idx = qa.Index(VARIANT)
cat_df = pd.read_csv(DATA_DIR / "catalog.csv", dtype=str) if (DATA_DIR / "catalog.csv").exists() else pd.DataFrame()

# 1차 자동화 JSON 저장 경로 & 최종 Human 평가 CSV 저장 경로
AUTO_JSON_PATH = VIEWS_DIR / "eval_50_auto_results.json"
FINAL_CSV_PATH = VIEWS_DIR / "eval_50_human_final.csv"
FINAL_JSONL_PATH = VIEWS_DIR / "kosis_rag_eval_50_reviewed.jsonl"

print(f"✅ 인덱스 로드 완료: {VARIANT} ({len(idx.ids):,}개 문서)")
print(f"📁 1차 자동화 JSON 경로 : {AUTO_JSON_PATH}")
print(f"📁 최종 Human 저장 경로 : {FINAL_CSV_PATH}")

✅ 인덱스 로드 완료: base (9,641개 문서)
📁 1차 자동화 JSON 경로 : kosis_data\rag_views\eval_50_auto_results.json
📁 최종 Human 저장 경로 : kosis_data\rag_views\eval_50_human_final.csv


--- 
## 🤖 [Phase 1] 1차 자동화: CSV 내 50개 쿼리 전체 조회 및 근거 자료 포함 JSON 저장
- 이 셀을 실행하면 `kosis_rag_eval_50.csv`의 50문항을 순회하며 **질의 재작성 ➔ 검색 ➔ 조건 슬라이싱 ➔ 답변 생성 ➔ 5대 지표 자동 채점**을 수행하고, **사람이 눈으로 바로 대조할 수 있는 근거 자료(Facts 요약, 조건 적용 표, Preview 표, 주의사항, KOSIS 웹 링크, 로컬 CSV 경로)**를 모두 묶어 `eval_50_auto_results.json`에 저장합니다.
- 중간에 멈춰도 이미 저장된 문항은 건너뛰고 이어서 실행됩니다.

In [3]:
# [Cell 2] 1차 자동화 실행기: 50문항 조회 + 근거 자료(Evidence) 패키징 -> JSON 저장

FORCE_REBUILD_JSON = False  # True로 바꾸면 1번 문항부터 새로 조회

def load_eval_50_source():
    for p in [Path("kosis_rag_eval_50.csv"), Path("kosis_rag_eval_50.jsonl"), DATA_DIR / "kosis_rag_eval_50.csv"]:
        if p.exists():
            lines = [l.strip() for l in p.read_text(encoding="utf-8-sig").splitlines() if l.strip()]
            if lines and lines[0].startswith("{"):
                return [json.loads(l) for l in lines]
            return pd.read_csv(p, dtype=str).fillna("").to_dict(orient="records")
    raise FileNotFoundError("kosis_rag_eval_50.csv 파일을 찾을 수 없습니다.")

def get_ref_info(rec):
    src = rec.get("source", {})
    org_id, tbl_id = src.get("org_id", ""), src.get("tbl_id", "")
    tbl_key = src.get("table_key", f"{org_id}_{tbl_id}")
    kosis_url = src.get("kosis_url") or f"https://kosis.kr/statHtml/statHtml.do?orgId={org_id}&tblId={tbl_id}"
    category = rec.get("category") or rec.get("survey_nm", "")
    raw_csv_path = DATA_DIR / "raw" / kv.safe(category) / f"{tbl_key}.csv"
    if not raw_csv_path.exists() and not cat_df.empty:
        m = cat_df[(cat_df["org_id"] == str(org_id)) & (cat_df["tbl_id"] == str(tbl_id))]
        if not m.empty:
            raw_csv_path = DATA_DIR / "raw" / kv.safe(m.iloc[0]["category"]) / f"{tbl_key}.csv"
    return {
        "view_id": rec.get("id", ""),
        "title": rec.get("title", ""),
        "tbl_nm": rec.get("tbl_nm", ""),
        "survey_nm": category,
        "unit": (rec.get("filters") or {}).get("unit", ""),
        "prd_range": f"{(rec.get('filters') or {}).get('prd_from', '')}~{(rec.get('filters') or {}).get('prd_to', '')}",
        "kosis_url": kosis_url,
        "raw_csv": raw_csv_path.as_posix(),
        "csv_exists": raw_csv_path.exists(),
    }

def run_single_query_and_pack_evidence(item: dict, k: int = 5) -> dict:
    qid = item["id"]
    target_survey = item["survey"]
    test_type = item["test_type"]
    question = item["question"]
    pass_criteria = item["pass_criteria"]
    t0 = time.time()

    # 1) 질의 재작성
    try:
        rw, subs = qa.rewrite(question)
    except Exception:
        rw, subs = question, []

    # 2) 하이브리드 검색
    queries = [rw] + subs
    agg = {}
    weights = {q: 1.0 for q in queries}
    if question not in weights:
        weights[question] = 0.5

    for q, w in weights.items():
        try:
            s_hits = idx.search(q, k=k * 2)
        except Exception:
            bm_only = idx.search_bm25(q, k=k * 2)
            s_hits = [{"id": b["id"], "score": b["score"], "title": b["title"],
                       "doc_type": idx.by_id[b["id"]].get("doc_type"),
                       "bm25_rank": r_i, "dense_rank": None, "dense_sim": 0.0} for r_i, b in enumerate(bm_only)]
        for r_i, h in enumerate(s_hits):
            agg.setdefault(h["id"], {"h": h, "s": 0.0})["s"] += w / (qa.RRF_K + r_i + 1)

    hits = [v["h"] | {"fused": round(v["s"], 5)} for v in sorted(agg.values(), key=lambda x: -x["s"])][:k]

    # 3) 컨텍스트 및 조건 적용 근거 추출
    ctx_recs = []
    for h in hits:
        rec = idx.by_id[h["id"]]
        if rec.get("doc_type") == "table":
            vids = [v for v in rec.get("view_ids", []) if v in idx.by_id][:1]
            if vids and vids[0] not in [c["id"] for c in ctx_recs]:
                rec = idx.by_id[vids[0]]
        if rec["id"] not in [c["id"] for c in ctx_recs] and rec.get("doc_type") == "view":
            ctx_recs.append(rec)

    conds = {}
    cond_tables_for_human = {}  # 사람이 눈으로 볼 조건 필터링 결과 표
    for r in ctx_recs:
        try:
            src, applied, missing = qa.apply_conditions(r, question)
            if applied:
                df_c, crow = qa.condition_rows(src, max_rows=30)
                cond_tables_for_human[r["id"]] = df_c.head(15).to_dict(orient="records")
            else:
                crow = None
        except Exception as e:
            src, applied, missing, crow = r["source"], {}, [f"조건 적용 오류: {e.__class__.__name__}"], None
        conds[r["id"]] = (applied, missing, crow)

    ctx_recs.sort(key=lambda r: (len(conds[r["id"]][1]) > 0, -len(conds[r["id"]][0])))
    ctx_recs = ctx_recs[:4]
    full_context = "\n\n".join(qa.context_block(r, 40, conds[r["id"]]) for r in ctx_recs) if ctx_recs else ""

    asked_conds = qa.mentioned_regions(question) + re.findall(r"(19\d{2}|20\d{2})\s*년?", question)
    cond_ok = (not asked_conds) or any(not conds[r["id"]][1] and conds[r["id"]][0] for r in ctx_recs)

    # 4) LLM 답변 생성
    try:
        out = qa.chat(
            [{"role": "system", "content": qa.ANSWER_SYS},
             {"role": "user", "content": f"[질문]\n{question}\n\n[검색어]\n{' / '.join(queries)}\n\n[자료]\n{full_context}"}],
            schema=qa.ANSWER_SCHEMA,
            max_tokens=1200,
        )
        ans_text = out.get("answer") or out.get("_raw", "")
        summary_text = out.get("summary", "")
        key_points = out.get("key_points") or []
        used_ids = [u for u in (out.get("used_ids") or []) if u in idx.by_id] or ([ctx_recs[0]["id"]] if ctx_recs else [])
        insufficient = bool(out.get("insufficient")) or not cond_ok
    except Exception as e:
        ans_text = f"[API 호출 오류: {e}]"
        summary_text, key_points = "", []
        used_ids = [ctx_recs[0]["id"]] if ctx_recs else []
        insufficient = not cond_ok

    unverified = qa.verify_numbers(" ".join([ans_text, summary_text] + list(key_points)), full_context) if full_context else []

    # 5) 5대 자동화 지표 판정 (PASS / PARTIAL / FAIL) 및 판정 사유 생성
    top_rec = idx.by_id[used_ids[0]] if (used_ids and used_ids[0] in idx.by_id) else (ctx_recs[0] if ctx_recs else {})
    retrieved_surveys = {idx.by_id[h["id"]].get("category") or idx.by_id[h["id"]].get("survey_nm", "") for h in hits}
    expected_surveys = set(target_survey.split("+")) if target_survey != "범위외" else set()

    # 지표 1: 검색 적합성
    if test_type == "NEGATIVE" or target_survey == "범위외":
        m_ret = "PASS" if insufficient else "PARTIAL"
        r_ret = f"범위외 질문 -> insufficient={insufficient}"
    else:
        matched_sv = expected_surveys & retrieved_surveys
        if len(matched_sv) == len(expected_surveys) and expected_surveys:
            m_ret, r_ret = "PASS", f"목표 조사({target_survey}) 정확히 매칭됨"
        elif matched_sv:
            m_ret, r_ret = "PARTIAL", f"복합 조사 중 일부만 매칭됨 ({matched_sv})"
        else:
            m_ret, r_ret = "FAIL", f"목표({target_survey}) 불일치 (검색됨: {retrieved_surveys})"

    # 지표 2: 원본 수치 추출
    facts_list = top_rec.get("facts") or []
    has_subtotal_bug = any(re.search(r"동부|읍부|면부|시부|군부", f.get("text", "")) for f in facts_list)
    if test_type == "NEGATIVE":
        m_ext, r_ext = "PASS", "범위외 질문 (수치 추출 불필요)"
    elif not ctx_recs:
        m_ext, r_ext = "FAIL", "추출된 View 없음"
    elif has_subtotal_bug:
        m_ext, r_ext = "PARTIAL", "⚠ Facts 내 도농 소계(동부/읍부/면부) 혼입 감지"
    elif facts_list or cond_tables_for_human.get(top_rec.get("id", "")):
        m_ext, r_ext = "PASS", f"원본 수치 정상 추출 ({len(facts_list)}개 핵심수치)"
    else:
        m_ext, r_ext = "PARTIAL", "기본 미리보기만 존재"

    # 지표 3: Context 충분성
    early_return_lost = bool(facts_list) and ("핵심 수치(기본 단면):" not in full_context)
    if test_type == "NEGATIVE":
        m_ctx, r_ctx = "PASS", "범위외 질문"
    elif not cond_ok or len(full_context) < 50:
        missing_str = "; ".join(conds[top_rec["id"]][1]) if top_rec.get("id") in conds else "컨텍스트 부족"
        m_ctx, r_ctx = "FAIL", f"조건 불일치/누락: {missing_str}"
    elif early_return_lost and test_type in ("CALC", "TREND"):
        m_ctx, r_ctx = "PARTIAL", "⚠ 조건 적용(Early Return)으로 인해 사전 계산된 변화량/변화율(facts)이 프롬프트에서 생략됨"
    else:
        m_ctx, r_ctx = "PASS", "질문 조건 및 수치 컨텍스트 충분"

    # 지표 4: 변화율 응답 완성도
    ans_nums = qa.nums(ans_text + " " + " ".join(key_points))
    if ans_text.startswith("[API 호출 오류"):
        m_comp, r_comp = "FAIL", "API 호출 실패"
    elif test_type == "NEGATIVE":
        ok_neg = insufficient or bool(re.search(r"없습니다|어렵습니다|수치.*없|추계|확인할 수 없|개인.*식별|범위", ans_text))
        m_comp, r_comp = ("PASS", "범위외 한계 정상 안내") if ok_neg else ("FAIL", "범위외 질문에 부적절 응답")
    elif test_type in ("CALC", "TREND"):
        has_pct = bool(re.search(r"%|퍼센트|증감률|변화율|비중|비율", ans_text))
        if has_pct and not unverified:
            m_comp, r_comp = "PASS", "수치 및 증감/비율(%) 계산 완전 일치"
        elif len(ans_nums) >= 2:
            m_comp, r_comp = "PARTIAL", f"원시 수치는 제시했으나 변화율(%) 누락 또는 암산 오차({unverified})"
        else:
            m_comp, r_comp = "FAIL", "핵심 비교 수치 및 변화율 모두 누락"
    else:
        if not insufficient and len(ans_nums) >= 1 and not has_subtotal_bug:
            m_comp, r_comp = "PASS", "요구 수치 및 단위 정상 응답"
        elif insufficient and test_type == "COVERAGE":
            m_comp, r_comp = "PASS", "미수록/불충분 여부 정상 안내"
        elif len(ans_nums) >= 1:
            m_comp, r_comp = "PARTIAL", "부분 응답 또는 소계 혼입·조건 불일치 안내"
        else:
            m_comp, r_comp = "FAIL", "유효 수치 미응답"

    # 지표 5: 자동 환각 검증
    if ans_text.startswith("[API 호출 오류"):
        m_hal, r_hal = "FAIL", "API 오류"
    elif not unverified:
        m_hal, r_hal = "PASS", "미검증 숫자 없음 (100% 컨텍스트 일치)"
    elif test_type in ("CALC", "TREND") and len(unverified) <= 2:
        m_hal, r_hal = "PARTIAL", f"LLM 직접 연산 파생 숫자 감지: {unverified} (원본과 대조 필요)"
    else:
        m_hal, r_hal = "FAIL", f"컨텍스트에 없는 미검증 수치 다수 포함: {unverified}"

    grades = [m_ret, m_ext, m_ctx, m_comp, m_hal]
    auto_overall = "PASS" if all(g == "PASS" for g in grades) else ("FAIL" if "FAIL" in (m_ret, m_comp) else "PARTIAL")

    # 6) 사람이 바로 보고 판단할 수 있도록 근거 자료(Evidence) 패키징
    evidence_sources = []
    for r in ctx_recs[:2]:
        rid = r["id"]
        applied_c, missing_c, _ = conds.get(rid, ({}, [], None))
        pv = r.get("preview") or {}
        pv_records = [dict(zip(pv.get("columns", []), row_v)) for row_v in (pv.get("rows") or [])[:12]]
        evidence_sources.append({
            **get_ref_info(r),
            "description": r.get("description", ""),
            "applied_conditions": applied_c,
            "missing_conditions": missing_c,
            "facts_summary": [f["text"] for f in (r.get("facts") or [])],
            "condition_filtered_rows": cond_tables_for_human.get(rid, []),
            "preview_rows": pv_records,
            "caveats": (r.get("caveats") or [])[:2],
        })

    return {
        "id": qid,
        "survey": target_survey,
        "test_type": test_type,
        "target_modules": item.get("target_modules", ""),
        "question": question,
        "pass_criteria": pass_criteria,
        "rewritten_query": rw,
        "ai_answer": ans_text,
        "ai_summary": summary_text,
        "ai_key_points": key_points,
        "insufficient": insufficient,
        "unverified_numbers": unverified,
        "auto_metrics": {
            "검색_적합성": m_ret,
            "원본_수치_추출": m_ext,
            "Context_충분성": m_ctx,
            "변화율_응답_완성도": m_comp,
            "자동_환각_검증": m_hal,
            "자동_종합_판정": auto_overall,
        },
        "auto_reasons": {
            "검색_적합성": r_ret,
            "원본_수치_추출": r_ext,
            "Context_충분성": r_ctx,
            "변화율_응답_완성도": r_comp,
            "자동_환각_검증": r_hal,
        },
        "evidence_sources": evidence_sources,
        "prompt_context_raw": full_context,
        "human_review": {
            "검색_적합성": m_ret,
            "원본_수치_추출": m_ext,
            "Context_충분성": m_ctx,
            "변화율_응답_완성도": m_comp,
            "자동_환각_검증": m_hal,
            "최종_HUMAN_평가": auto_overall,  # 사람이 Step 2에서 PASS / PARTIAL / FAIL로 최종 확정
            "human_comment": "",
            "reviewed": False,
        },
        "elapsed_s": round(time.time() - t0, 2),
    }

# --- 배치 실행 및 JSON 저장 ---
eval_items = load_eval_50_source()
saved_data = {}
if AUTO_JSON_PATH.exists() and not FORCE_REBUILD_JSON:
    saved_data = {x["id"]: x for x in json.loads(AUTO_JSON_PATH.read_text(encoding="utf-8"))}
    print(f"🔄 기존 JSON에서 {len(saved_data)}건 로드됨. 미완료 문항만 실행합니다.")

for i, item in enumerate(eval_items, 1):
    qid = item["id"]
    if qid in saved_data and not str(saved_data[qid].get("ai_answer", "")).startswith("[API 호출 오류"):
        continue
    res = run_single_query_and_pack_evidence(item, k=5)
    saved_data[qid] = res
    # 매 문항마다 JSON 즉시 저장
    ordered_list = [saved_data[k_id] for k_id in sorted(saved_data.keys())]
    AUTO_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
    AUTO_JSON_PATH.write_text(json.dumps(ordered_list, ensure_ascii=False, indent=2), encoding="utf-8")
    m = res["auto_metrics"]
    print(f"[{i:02d}/50] {qid} ({res['survey']}) ➔ 검색:{m['검색_적합성']} | 추출:{m['원본_수치_추출']} | Ctx:{m['Context_충분성']} | 완성도:{m['변화율_응답_완성도']} | 환각:{m['자동_환각_검증']} 👉 [{m['자동_종합_판정']}] ({res['elapsed_s']}s)")

print(f"\n✅ 1차 자동화 완료! 근거 자료가 포함된 JSON 저장됨 ➔ {AUTO_JSON_PATH}")

[01/50] Q001 (인구총조사) ➔ 검색:PASS | 추출:PASS | Ctx:PASS | 완성도:PASS | 환각:FAIL 👉 [PARTIAL] (8.59s)
[02/50] Q002 (인구총조사) ➔ 검색:PASS | 추출:PARTIAL | Ctx:PASS | 완성도:PARTIAL | 환각:PASS 👉 [PARTIAL] (4.46s)
[03/50] Q003 (인구총조사) ➔ 검색:PASS | 추출:PASS | Ctx:PARTIAL | 완성도:PARTIAL | 환각:PARTIAL 👉 [PARTIAL] (6.54s)
[04/50] Q004 (인구총조사) ➔ 검색:PASS | 추출:PASS | Ctx:PASS | 완성도:PARTIAL | 환각:FAIL 👉 [PARTIAL] (6.55s)
[05/50] Q005 (인구총조사+주민등록인구현황) ➔ 검색:PASS | 추출:PASS | Ctx:PASS | 완성도:PASS | 환각:FAIL 👉 [PARTIAL] (7.42s)
[06/50] Q006 (인구동향조사) ➔ 검색:PASS | 추출:PASS | Ctx:PASS | 완성도:PASS | 환각:PASS 👉 [PASS] (5.47s)
[07/50] Q007 (인구동향조사) ➔ 검색:PASS | 추출:PASS | Ctx:PARTIAL | 완성도:PARTIAL | 환각:PARTIAL 👉 [PARTIAL] (6.16s)
[08/50] Q008 (인구동향조사) ➔ 검색:PASS | 추출:PASS | Ctx:PARTIAL | 완성도:PARTIAL | 환각:PASS 👉 [PARTIAL] (4.94s)
[09/50] Q009 (인구동향조사) ➔ 검색:PASS | 추출:PASS | Ctx:PASS | 완성도:PARTIAL | 환각:PASS 👉 [PARTIAL] (4.57s)
[10/50] Q010 (영주귀국사할린동포실태조사) ➔ 검색:PASS | 추출:PASS | Ctx:PASS | 완성도:PASS | 환각:PASS 👉 [PASS] (5.96s)
[11/50] Q011 (영주귀국사

--- 
## 👀 [Phase 2 - 쉘 1] 저장된 JSON 불러오기 & 전체 50문항 신호등 요약표 보기
- 1차 자동화로 생성된 `eval_50_auto_results.json`을 불러와서 **어느 문항이 `PARTIAL`이나 `FAIL`인지 컬러 배지로 한눈에 확인**합니다.

In [4]:
# [Cell 3] JSON 로드 및 한눈에 보는 컬러 배지 요약 테이블
assert AUTO_JSON_PATH.exists(), "먼저 위 [Cell 2]를 실행하여 JSON 파일을 생성하세요."
auto_results = json.loads(AUTO_JSON_PATH.read_text(encoding="utf-8"))
results_by_id = {r["id"]: r for r in auto_results}

def badge(val):
    colors = {
        "PASS": "background-color:#d4edda; color:#155724; font-weight:bold; padding:3px 8px; border-radius:6px;",
        "PARTIAL": "background-color:#fff3cd; color:#856404; font-weight:bold; padding:3px 8px; border-radius:6px;",
        "FAIL": "background-color:#f8d7da; color:#721c24; font-weight:bold; padding:3px 8px; border-radius:6px;",
    }
    st = colors.get(val, "padding:3px 8px;")
    return f'<span style="{st}">{val}</span>'

# 필터링 옵션: 전체 보기("ALL") 또는 문제 있는 문항만 빠르게 보기("PARTIAL_OR_FAIL")
VIEW_FILTER = "ALL"  # "ALL" | "PARTIAL_OR_FAIL" | "UNREVIEWED"

overview_rows = []
for r in auto_results:
    hm = r["human_review"]
    if VIEW_FILTER == "PARTIAL_OR_FAIL" and hm["최종_HUMAN_평가"] == "PASS":
        continue
    if VIEW_FILTER == "UNREVIEWED" and hm.get("reviewed"):
        continue
    overview_rows.append({
        "ID": f"<b>{r['id']}</b>",
        "조사명": r["survey"],
        "유형": r["test_type"],
        "질문 요약": r["question"][:42] + "...",
        "검색 적합성": badge(hm["검색_적합성"]),
        "원본 수치 추출": badge(hm["원본_수치_추출"]),
        "Context 충분성": badge(hm["Context_충분성"]),
        "변화율 응답 완성도": badge(hm["변화율_응답_완성도"]),
        "자동 환각 검증": badge(hm["자동_환각_검증"]),
        "최종 HUMAN 평가": badge(hm["최종_HUMAN_평가"]),
        "검수상태": "✅ 완료" if hm.get("reviewed") else "⏳ 대기",
        "코멘트": hm.get("human_comment", "")[:30],
    })

print(f"📊 전체 {len(auto_results)}문항 중 표시된 문항: {len(overview_rows)}건")
display(HTML(pd.DataFrame(overview_rows).to_html(escape=False, index=False)))

📊 전체 50문항 중 표시된 문항: 50건


ID,조사명,유형,질문 요약,검색 적합성,원본 수치 추출,Context 충분성,변화율 응답 완성도,자동 환각 검증,최종 HUMAN 평가,검수상태,코멘트
Q001,인구총조사,FILTER,"2020년 인구총조사 기준 서울특별시의 남자와 여자 인구는 각각 몇 명이고,...",PASS,PASS,PASS,PASS,FAIL,PARTIAL,⏳ 대기,
Q002,인구총조사,RANKING,2005년 인구총조사의 '행정구역별 1세대가구-계'에서 시도별 가장 많은 곳...,PASS,PARTIAL,PASS,PARTIAL,PASS,PARTIAL,⏳ 대기,
Q003,인구총조사,CALC,"2015년과 2020년 전국의 1인 가구 수는 각각 얼마고, 5년간 몇 가구...",PASS,PASS,PARTIAL,PARTIAL,PARTIAL,PARTIAL,⏳ 대기,
Q004,인구총조사,FILTER,2020년 부산광역시의 65세 이상 인구와 0~14세 인구는 각각 몇 명이고...,PASS,PASS,PASS,PARTIAL,FAIL,PARTIAL,⏳ 대기,
Q005,인구총조사+주민등록인구현황,CROSS,2020년 서울 총인구를 인구총조사 기준과 주민등록인구현황의 12월 말 기준...,PASS,PASS,PASS,PASS,FAIL,PARTIAL,⏳ 대기,
Q006,인구동향조사,SEMANTIC,2023년 전국의 출생아 수와 합계출산율을 각각 알려 줘. 두 지표의 단위도...,PASS,PASS,PASS,PASS,PASS,PASS,⏳ 대기,
Q007,인구동향조사,TREND,"2020~2024년 전국 출생아 수를 연도별로 보여 주고, 2020년 대비 ...",PASS,PASS,PARTIAL,PARTIAL,PARTIAL,PARTIAL,⏳ 대기,
Q008,인구동향조사,CALC,2023년 전국의 자연증가 인구는 몇 명이야? 출생아 수에서 사망자 수를 뺀...,PASS,PASS,PARTIAL,PARTIAL,PASS,PARTIAL,⏳ 대기,
Q009,인구동향조사,FILTER,2023년 서울특별시의 혼인건수와 이혼건수는 각각 몇 건이야? 전국 값이 섞...,PASS,PASS,PASS,PARTIAL,PASS,PARTIAL,⏳ 대기,
Q010,영주귀국사할린동포실태조사,COVERAGE,영주귀국사할린동포실태조사에서 현재 수록된 가장 최근 조사연도와 그때의 조사대...,PASS,PASS,PASS,PASS,PASS,PASS,⏳ 대기,


--- 
## 2 - 사람이 빠르게 보고 판단하는 『초간편 시각적 검수 카드 & 즉시 저장기』
- 아래 셀에서 **검수할 문항 번호(`QID`)**와 **최종 판정(`PASS` / `PARTIAL` / `FAIL`) 및 한 줄 코멘트**만 적고 `Ctrl + Enter`를 누르면:
  1. **질문 & 통과 기준(`pass_criteria`)**
  2. **AI 답변 & 미검증 숫자(`unverified_numbers`) 하이라이트**
  3. **5대 자동화 지표 배지 & 자동 진단 이유**
  4. **근거 자료 카드 (KOSIS 바로가기 링크 + 로컬 CSV 경로 + Facts 핵심 수치 + 실제 조건 필터링 표)**
  가 한 화면에 깔끔하게 출력되고, 입력한 사람의 평가가 JSON과 최종 CSV(`eval_50_human_final.csv`)에 즉시 반영됩니다.

In [5]:
# [Cell 4] 문항별 시각적 근거 카드 출력 함수 정의
def show_visual_review_card(qid: str):
    r = results_by_id[qid]
    am = r["auto_metrics"]
    ar = r["auto_reasons"]
    hm = r["human_review"]
    ev_list = r.get("evidence_sources", [])

    # 근거 자료 HTML 조립
    ev_blocks_html = ""
    for idx_e, ev in enumerate(ev_list, 1):
        facts_html = "<br>".join(f"• {f}" for f in ev.get("facts_summary", [])) or "(요약 Facts 없음)"
        
        # 조건 적용된 실제 데이터 표 또는 미리보기 표를 작은 HTML 테이블로 렌더링
        cond_rows = ev.get("condition_filtered_rows") or []
        pv_rows = ev.get("preview_rows") or []
        if cond_rows:
            tbl_label = f"📌 질문 조건 동적 필터링 표 (적용 조건: {ev.get('applied_conditions')})"
            mini_df_html = pd.DataFrame(cond_rows).head(10).to_html(index=False, border=1)
        elif pv_rows:
            tbl_label = "📌 View 기본 미리보기 표 (상위 8행)"
            mini_df_html = pd.DataFrame(pv_rows).head(8).to_html(index=False, border=1)
        else:
            tbl_label = "📌 표 데이터 없음"
            mini_df_html = ""

        missing_warn = (
            f'<div style="color:#721c24; background:#f8d7da; padding:4px 8px; border-radius:4px; margin-top:4px;">'
            f'⚠️ 누락/불일치 조건: {", ".join(ev["missing_conditions"])}</div>'
            if ev.get("missing_conditions") else ""
        )

        ev_blocks_html += f"""
        <div style="border:1px solid #cbd5e1; border-radius:8px; padding:12px; margin-top:10px; background:#f8fafc;">
            <div style="font-weight:bold; font-size:14px; color:#1e293b;">
                📂 [근거 자료 #{idx_e}] {ev['title']} <span style="color:#64748b; font-size:12px;">({ev['view_id']})</span>
            </div>
            <div style="font-size:12px; color:#475569; margin:4px 0;">
                • <b>조사명/통계표</b>: {ev['survey_nm']} &gt; {ev['tbl_nm']} (수록기간: {ev['prd_range']}, 단위: {ev['unit'] or '없음'})<br>
                • <b>원본 대조 링크</b>: <a href="{ev['kosis_url']}" target="_blank" style="font-weight:bold; color:#2563eb;">🌐 KOSIS 웹 통계표 바로가기</a> 
                  | 📁 로컬 CSV: <code>{ev['raw_csv']}</code> ({'✅ 존재' if ev['csv_exists'] else '❌ 없음'})
            </div>
            {missing_warn}
            <div style="display:flex; gap:16px; margin-top:8px; align-items:flex-start;">
                <div class="facts-box" style="flex:1; background:#ffffff; padding:8px; border:1px solid #000000; border-radius:6px; font-size:12px; color:#000000;">
                    <b>📊 원본 추출 핵심 수치 (Facts 정답 기준)</b><br>{facts_html}
                </div>
                <div style="flex:1.3; background:#ffffff; padding:8px; border:1px solid #000000; border-radius:6px; font-size:12px; overflow-x:auto;">
                    <b>{tbl_label}</b>{mini_df_html}
                </div>
            </div>
        </div>
        """

    unv_str = ", ".join(r.get("unverified_numbers") or [])
    unv_html = (
        f'<span style="background:#fee2e2; color:#991b1b; padding:2px 6px; border-radius:4px; font-weight:bold;">⚠️ 미검증/오산 의심 수치: {unv_str}</span>'
        if unv_str else '<span style="color:#15803d; font-weight:bold;">✅ 미검증 수치 없음 (컨텍스트 내 숫자와 일치)</span>'
    )

    card_html = f"""
    <div class="visual-review-card" style="border:2px solid #334155; border-radius:10px; padding:16px; background:#ffffff; font-family:sans-serif; color:#000000;">
        <div style="display:flex; justify-content:space-between; align-items:center; border-bottom:2px solid #000000; padding-bottom:10px;">
            <div>
                <span style="font-size:18px; font-weight:bold; color:#0f172a;">[{r['id']}] {r['survey']}</span>
                <span style="background:#e2e8f0; color:#000000; padding:3px 8px; border-radius:4px; font-size:12px; margin-left:8px;">유형: {r['test_type']}</span>
                <span style="background:#e0e7ff; color:#1e40af; padding:3px 8px; border-radius:4px; font-size:12px; margin-left:4px;">타겟 모듈: {r['target_modules']}</span>
            </div>
            <div style="font-size:14px;">
                최종 HUMAN 판정: {badge(hm['최종_HUMAN_평가'])} {'(✅ 검수완료)' if hm.get('reviewed') else '(⏳ 자동초안)'}
            </div>
        </div>

        <!-- 질문 및 통과 기준 -->
        <div style="margin-top:12px; background:#f1f5f9; padding:10px 14px; border-radius:8px;">
            <div style="font-size:15px; font-weight:bold; color:#0f172a;">❓ 입력 질문: {r['question']}</div>
            <div style="font-size:13px; color:#b45309; margin-top:4px;">🎯 <b>채점 통과 기준 (pass_criteria)</b>: {r['pass_criteria']}</div>
            <div style="font-size:12px; color:#64748b; margin-top:2px;">🔎 재작성 검색어: <code>{r['rewritten_query']}</code></div>
        </div>

        <!-- AI 응답 -->
        <div style="margin-top:10px; background:#eff6ff; border-left:4px solid #2563eb; padding:10px 14px; border-radius:4px;">
            <div style="font-size:14px; font-weight:bold; color:#1e3a8a;">🤖 AI 출력 답변:</div>
            <div style="font-size:14px; color:#1e293b; margin-top:4px; line-height:1.5;">{r['ai_answer']}</div>
            <div style="font-size:12px; color:#475569; margin-top:6px;">
                • <b>핵심 수치(key_points)</b>: {', '.join(r.get('ai_key_points') or []) or '없음'}<br>
                • <b>수치 검증 상태</b>: {unv_html} | <b>자료부족(insufficient) 플래그</b>: <code>{r['insufficient']}</code>
            </div>
        </div>

        <!-- 5대 지표 테이블 -->
        <table style="width:100%; border-collapse:collapse; margin-top:12px; font-size:13px;">
            <thead>
                <tr style="background:#1e293b; color:#ffffff; text-align:center;">
                    <th style="padding:6px; width:18%;">5대 핵심 지표</th>
                    <th style="padding:6px; width:13%;">1차 자동 판정</th>
                    <th style="padding:6px; width:13%;">현재 HUMAN 판정</th>
                    <th style="padding:6px; text-align:left;">자동 진단 근거 (요약)</th>
                </tr>
            </thead>
            <tbody>
                <tr style="border-bottom:1px solid #000000; text-align:center;">
                    <td style="padding:6px; font-weight:bold;">1. 검색 적합성</td>
                    <td>{badge(am['검색_적합성'])}</td>
                    <td>{badge(hm['검색_적합성'])}</td>
                    <td style="text-align:left; padding:6px;">{ar['검색_적합성']}</td>
                </tr>
                <tr style="border-bottom:1px solid #000000; text-align:center;">
                    <td style="padding:6px; font-weight:bold;">2. 원본 수치 추출</td>
                    <td>{badge(am['원본_수치_추출'])}</td>
                    <td>{badge(hm['원본_수치_추출'])}</td>
                    <td style="text-align:left; padding:6px;">{ar['원본_수치_추출']}</td>
                </tr>
                <tr style="border-bottom:1px solid #000000; text-align:center;">
                    <td style="padding:6px; font-weight:bold;">3. Context 충분성</td>
                    <td>{badge(am['Context_충분성'])}</td>
                    <td>{badge(hm['Context_충분성'])}</td>
                    <td style="text-align:left; padding:6px;">{ar['Context_충분성']}</td>
                </tr>
                <tr style="border-bottom:1px solid #000000; text-align:center;">
                    <td style="padding:6px; font-weight:bold;">4. 변화율 응답 완성도</td>
                    <td>{badge(am['변화율_응답_완성도'])}</td>
                    <td>{badge(hm['변화율_응답_완성도'])}</td>
                    <td style="text-align:left; padding:6px;">{ar['변화율_응답_완성도']}</td>
                </tr>
                <tr style="border-bottom:1px solid #000000; text-align:center;">
                    <td style="padding:6px; font-weight:bold;">5. 자동 환각 검증</td>
                    <td>{badge(am['자동_환각_검증'])}</td>
                    <td>{badge(hm['자동_환각_검증'])}</td>
                    <td style="text-align:left; padding:6px;">{ar['자동_환각_검증']}</td>
                </tr>
            </tbody>
        </table>

        <!-- 근거 자료 블록 -->
        {ev_blocks_html}

        <div style="margin-top:10px; background:#fefce8; border:1px solid #fde047; padding:8px 12px; border-radius:6px; font-size:13px;">
            ✍️ <b>사람 검수 코멘트</b>: {hm.get('human_comment') or '(아직 작성된 코멘트가 없습니다)'}
        </div>
    </div>
    """
    

    # HTML에 명시된 회색 텍스트를 검은색으로 변경
    gray_colors = [
        "#64748b",
        "#475569",
        "#334155",
        "#1e293b",
    ]

    for gray in gray_colors:
        card_html = card_html.replace(
            f"color:{gray};",
            "color:#000000;"
        )

    # Jupyter 기본 스타일보다 우선 적용할 CSS
    custom_css = """
    <style>
        /* 카드의 기본 글자색 */
        .visual-review-card {
            color: #000000 !important;
        }

        /* 5대 지표 및 원본 데이터 표의 셀 */
        .visual-review-card table tbody td,
        .visual-review-card table tbody th {
            color: #000000 !important;
        }

        /* 원본 Facts와 미리보기 영역 */
        .visual-review-card .facts-box,
        .visual-review-card .preview-box {
            color: #000000 !important;
        }

        /* 미리보기 데이터 표의 헤더 */
        .visual-review-card .preview-box table th {
            color: #000000 !important;
        }

        /* 코드 및 경로 텍스트 */
        .visual-review-card code {
            color: #000000 !important;
        }
    </style>
    """
    final_html = custom_css + card_html
    display(HTML(final_html))
    return final_html

def save_all_human_reviews():
    """현재 메모리의 results_by_id를 JSON, 최종 CSV, 원본 호환 JSONL 3곳에 동시 저장"""
    ordered = [results_by_id[k] for k in sorted(results_by_id.keys())]
    AUTO_JSON_PATH.write_text(json.dumps(ordered, ensure_ascii=False, indent=2), encoding="utf-8")

    csv_rows = []
    for r in ordered:
        hm = r["human_review"]
        ev0 = (r.get("evidence_sources") or [{}])[0]
        csv_rows.append({
            "id": r["id"],
            "survey": r["survey"],
            "test_type": r["test_type"],
            "target_modules": r["target_modules"],
            "question": r["question"],
            "ai_answer": r["ai_answer"],
            " 근거_Facts_요약": " | ".join(ev0.get("facts_summary", [])),
            "KOSIS_링크": ev0.get("kosis_url", ""),
            "로컬_CSV": ev0.get("raw_csv", ""),
            "검색_적합성": hm["검색_적합성"],
            "원본_수치_추출": hm["원본_수치_추출"],
            "Context_충분성": hm["Context_충분성"],
            "변화율_응답_완성도": hm["변화율_응답_완성도"],
            "자동_환각_검증": hm["자동_환각_검증"],
            "최종_HUMAN_평가": hm["최종_HUMAN_평가"],
            "human_comment": hm.get("human_comment", ""),
            "reviewed": hm.get("reviewed", False),
        })
    pd.DataFrame(csv_rows).to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")
print("✅ 시각적 검수 카드 렌더러(show_visual_review_card) 준비 완료!")

✅ 시각적 검수 카드 렌더러(show_visual_review_card) 준비 완료!


In [6]:
from html2image import Html2Image
hti = Html2Image()

In [41]:
# [Cell 5] 👇 여기서 문항 번호(QID)만 바꿔가며 카드를 보고 빠르게 최종 평가 및 코멘트를 저장하세요!

QID = "Q015"  # Q001 ~ Q050 입력

# 1) 먼저 현재 문항의 시각적 근거 카드를 보고 싶다면 아래 값을 None으로 두고 실행하세요.
# 2) 판정을 수정/확정하려면 "PASS", "PARTIAL", "FAIL" 중 하나와 코멘트를 적고 실행하세요.
FINAL_VERDICT = "PAPARTIALSS"          # "PASS" | "PARTIAL" | "FAIL" | None (None이면 자동 판정값 유지)
COMMENT_TEXT  = ""
# 세부 지표를 개별 수정하고 싶을 때만 변경 (None이면 자동화 지표 그대로 유지)
DETAIL_OVERRIDES = {
    "검색_적합성": "PARTIAL",        # "PASS" | "PARTIAL" | "FAIL" | None
    "원본_수치_추출": "PASS",     # "PASS" | "PARTIAL" | "FAIL" | None
    "Context_충분성": "PARTIAL",    # "PASS" | "PARTIAL" | "FAIL" | None
    "변화율_응답_완성도": "PARTIAL", # "PASS" | "PARTIAL" | "FAIL" | None
    "자동_환각_검증": "PARTIAL",     # "PASS" | "PARTIAL" | "FAIL" | None
}

# --- 입력 반영 및 즉시 저장 ---
hm = results_by_id[QID]["human_review"]
for k_col, v_col in DETAIL_OVERRIDES.items():
    if v_col in ("PASS", "PARTIAL", "FAIL"):
        hm[k_col] = v_col
        hm["reviewed"] = True

if FINAL_VERDICT in ("PASS", "PARTIAL", "FAIL"):
    hm["최종_HUMAN_평가"] = FINAL_VERDICT
    hm["reviewed"] = True

if COMMENT_TEXT is not None:
    hm["human_comment"] = COMMENT_TEXT
    hm["reviewed"] = True

save_all_human_reviews()
html_content = show_visual_review_card(QID)
print(f"💾 [{QID}] 현재 상태 저장 완료 ➔ {FINAL_CSV_PATH}")

save_dir = "review_images" 
os.makedirs(save_dir, exist_ok=True) 

# 2) 반환받은 HTML 문자열이 있으면 이미지로 저장
if html_content:
    hti = Html2Image()
    # hti.size = (800, 600) 
    save_filename = os.path.join(save_dir, f"{QID}_review_card.png")
    hti.output_path = save_dir
    hti.screenshot(html_str=html_content, save_as=f"{QID}_review_card.png")
    print(f"📸 이미지 저장 완료: {save_filename}")

💾 [Q015] 현재 상태 저장 완료 ➔ kosis_data\rag_views\eval_50_human_final.csv
📸 이미지 저장 완료: review_images\Q015_review_card.png
